# Country Missingness Scoring

Identifies countries systematically absent from data they should have.
Classifies each (country, year) into 10 statuses: dissolved, political_exclusion,
self_exclusion, nascent, collision, microstate, failed, degraded, reporting, strong.

**Phase 1: Model Definition**

See `phase1/document/country_missingness_methodology.md` for full methodology.

In [ ]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

In [ ]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

## Run Pipeline

In [ ]:
result = run_country_missingness(df, meta_df)

## Status Distribution

In [ ]:
sort(combine(groupby(result.status, :country_status), nrow => :count), :count, rev=true)

## Dissolved States

In [ ]:
filter(r -> r.is_dissolved, result.profiles)[:, [:ident_ccodealp, :ident_cname, :country_birth_year, :country_death_year, :dissolution_year]]

## Political Exclusion

Entities excluded BY others from international data programs (not state failure).

In [ ]:
pol_excl = filter(r -> r.is_political_exclusion, result.profiles)
if nrow(pol_excl) > 0
    for r in eachrow(pol_excl)
        println("  $(r.ident_ccodealp) $(r.ident_cname) (from $(r.pol_excl_from))")
        # Show coverage trajectory
        rows = filter(row -> row.ident_ccode == r.ident_ccode, result.status)
        for decade in [1990, 2000, 2010, 2020]
            dec = filter(row -> decade <= row.ident_year < decade + 10, rows)
            nrow(dec) == 0 && continue
            avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
            sts = join(unique(dec.country_status), "/")
            println("    $(decade)s: $sts ($avg% avg)")
        end
    end
else
    println("  None")
end

## Self-Exclusion

Entities excluding THEMSELVES from international data engagement.

In [ ]:
self_excl = filter(r -> r.is_self_exclusion, result.profiles)
if nrow(self_excl) > 0
    for r in eachrow(self_excl)
        println("  $(r.ident_ccodealp) $(r.ident_cname) (from $(r.self_excl_from))")
        rows = filter(row -> row.ident_ccode == r.ident_ccode, result.status)
        for decade in [1960, 1970, 1980, 1990, 2000, 2010, 2020]
            dec = filter(row -> decade <= row.ident_year < decade + 10, rows)
            nrow(dec) == 0 && continue
            avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
            sts = join(unique(dec.country_status), "/")
            println("    $(decade)s: $sts ($avg% avg)")
        end
    end
else
    println("  None")
end

## Microstates (pop < 100K)

In [ ]:
micros = filter(r -> r.is_microstate, result.profiles)
println("$(nrow(micros)) microstates:")
for r in eachrow(micros)
    pop_str = ismissing(r.max_pop) ? "?" : "$(Int(round(r.max_pop)))K"
    println("  $(r.ident_ccodealp) $(rpad(r.ident_cname, 30)) pop $pop_str")
end

## Failed Countries

Active countries with <40% global slug coverage AND >20ppt below subregion peers.
Should NOT include dissolved, nascent, collision, or exclusion entities.

In [ ]:
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
println("Countries with 'failed' years: $(length(failed_countries))\n")
for ccode in failed_countries
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    fyears = sort(filter(r -> r.country_status == "failed", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $(length(fyears)) failed years: $(first(fyears))–$(last(fyears))")
    for decade in [1960, 1970, 1980, 1990, 2000, 2010, 2020]
        dec = filter(r -> decade <= r.ident_year < decade + 10, rows)
        nrow(dec) == 0 && continue
        avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
        sts = join(unique(dec.country_status), "/")
        println("    $(decade)s: $sts ($avg% avg)")
    end
    println()
end

## Degraded Countries

Coverage dropping relative to self AND peers (not just global reporting lag).

In [ ]:
degraded = filter(r -> r.country_status == "degraded", result.status)
degraded_countries = unique(degraded.ident_ccode)
println("Countries with 'degraded' years: $(length(degraded_countries))\n")
for ccode in degraded_countries
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    dyears = sort(filter(r -> r.country_status == "degraded", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $(length(dyears)) degraded years: $(first(dyears))–$(last(dyears))")
end

## Nascent Countries

First 5 years of data — newly independent, building data infrastructure.

In [ ]:
nascent = filter(r -> r.country_status == "nascent", result.status)
nascent_countries = unique(nascent.ident_ccode)
println("Countries with 'nascent' years: $(length(nascent_countries))")

## Revised Slug Penetration

Population-weighted penetration recalculated with clean denominator
(excluding dissolved, micro, failed, exclusion, collision country-years).

In [ ]:
original_global = count(r -> r.original_penetration >= 0.95, eachrow(result.penetration))
revised_global = count(r -> r.revised_penetration >= 0.95, eachrow(result.penetration))
println("Slugs ≥95% penetration:")
println("  Original denominator: $original_global")
println("  Revised denominator:  $revised_global")
println("  New globals:          $(revised_global - original_global)")

In [ ]:
# Top gainers
first(result.penetration, 20)

## Save Flags

In [ ]:
# CSV.write("data/country_missingness_flags.csv", result.flags)
# println("\u2705 Saved country_missingness_flags.csv")